[Guide Index](README.md)

# 05.03 SHACL shapes as classes

A class and a shape are different things.

A class, such as `ex:Person`, names a set of things: its instances are the nodes typed `ex:Person`. A class is often defined in an ontology and is described by a name (`rdfs:label`), a description (`rdfs:comment`) and subclass relationships, such as `ex:Student rdfs:subClassOf ex:Person`. 

A shape, such as `ex:PersonShape`, describes what the data should look like, for example that the targets of `ex:PersonShape` should have one name. A shape is usually defined in a shapes graph. The class and the shape are often linked by a target, `sh:targetClass`, stating that a given shape should be applied to all of the instances of a given class (e.g. `ex:PersonShape sh:targetClass ex:Person`).

SHACL 1.2 allows one resource to define both the class and the shape. For example, the resource `ex:Person` can be declared as a class and as a shape, allowing a single resource to carry the class's name, description and subclasses together with the constraints on its instances. Keeping them separate, as `ex:Person` and `ex:PersonShape`, continues to be allowed and a common practice.

SHACL 1.2 defines `sh:ShapeClass` as a subclass of both `sh:NodeShape` and `rdfs:Class`. A resource that is declared to be a `sh:ShapeClass` is therefore entailed to be both a `sh:NodeShape` and an `rdfs:Class`. 

SHACL recommends, but does not require, that a graph that uses `sh:ShapeClass` imports the SHACL vocabulary with `owl:imports <http://www.w3.org/ns/shacl#>`. This allows software that does not know SHACL to read the vocabulary and see that `sh:ShapeClass` is a subclass of `rdfs:Class`, so it knows `ex:Person` is a class. The SHACL engine of StarLayer has the SHACL vocabulary built in, so this import is not needed for it. A reasoner outside the engine does not know it: RDFS or OWL inference over a graph does not conclude that a `sh:ShapeClass` is an `rdfs:Class` unless the vocabulary is added to the graph, as the registered `shacl_vocabulary` graph (`starlayer.starontology.get_ontology_graph("shacl_vocabulary")`). Another common approach is to keep the ontology and the shapes graph separate and combine them before validation, for example with `owl:imports`. A shape whose IRI is a class in the ontology is then also a class in the combined shapes graph without declaring it a `sh:ShapeClass`.

**Class membership targeting.** Under SHACL 1.1 a shape targets a class by using `sh:targetClass`.  All nodes of a given type (or of one of its subclasses) are targets of the shape. SHACL 1.2 introduces a second way to target shapes by class. A shape also targets the members of a class when the shape is itself the class. No `sh:targetClass` is required.

**Where the statements are read.** The `rdf:type` statements are always read from the data graph. The `rdfs:subClassOf` statements are also read from the data graph by default, but can also be read from the shapes graph. The exception is `sh:closed sh:ByTypes` (section 05.03.3), which always reads the classes and the hierarchy from the shapes graph.

This guide covers SHACL 1.2 behavior for shapes that are also classes:

- a shape that is also a class targets its instances (section 05.03.1),
- the class hierarchy can be read from the shapes graph (section 05.03.2), and
- a closed shape can allow the properties its types declare (section 05.03.3).

The [validation guide](05.01-validating-with-shacl-1.2.ipynb) covers the rest of validating with SHACL 1.2.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import Namespace
from starlayer.graph import StarLayerGraph
from starlayer.graph.config import set_default_prefixes
from starlayer.shacl import StarShaclProcessor

# EX and SH are starlayer.graph Namespaces: their terms are pyoxigraph NamedNodes, used to look terms up in the results below
# (StarLayerGraphs, see section 05.01.2); the StarLayerGraph inputs are built by parsing Turtle.
EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

# Every StarLayerGraph created after this uses these prefixes, so results, queries and the report text use short names such as ex:alice
set_default_prefixes({
    "ex": "http://example.org/",
    "sh": "http://www.w3.org/ns/shacl#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "xsd": "http://www.w3.org/2001/XMLSchema#",
})


## 05.03.1 A shape that is also a class

A shape that is also a class targets its instances without an explicit `sh:targetClass`. 

If `ex:Person` is a `sh:ShapeClass` (or `ex:Person a rdfs:Class` and `ex:Person a sh:NodeShape`) then on validation every node typed `ex:Person` (and every node typed as a subclass of `ex:Person`) will be validated against the `ex:Person` shape. The shape needs no explicit `sh:targetClass ex:Person`. 

Below, `ex:Person` has no explicit targets but still validates the people and the students.  The `target_nodes()` method lists the three people it targets. `ex:Student` is known to be a subclass of `ex:Person` from the data graph. `ex:Alice` and `ex:Bob`, a student, have no name (do not conform), and `ex:Carol` has a name (conforms). `ex:NewYork` is an `ex:Place`, which is not `ex:Person` or a subclass of it, so it is not a target.

In [ ]:
shapes = StarLayerGraph()
# note that ex:Person has no sh:targetClass
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:Person a sh:ShapeClass ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Student rdfs:subClassOf ex:Person .
    ex:Alice a ex:Person .
    ex:Bob a ex:Student .
    ex:Carol a ex:Person ; ex:name "Carol" .
    ex:NewYork a ex:Place .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
result = myProcessor.validate(data_graph=data)

print("targets:", sorted(data.qname(n) for n in myProcessor.target_nodes(data_graph=data, shape_node=EX.Person)))
print("conforms:", result.conforms)
print("focus nodes in the report:", sorted(result.report_graph.qname(n) for n in result.report_graph.objects(None, SH.focusNode)))

targets: ['ex:Alice', 'ex:Bob', 'ex:Carol']
conforms: False
focus nodes in the report: ['ex:Alice', 'ex:Bob']


## 05.03.2 Where the class hierarchy is read from: `subclass_of_in_shapes_graph`

By default `sh:class`, `sh:targetClass` and the other checks of whether a node is an instance of a class read `rdfs:subClassOf` from the data graph. A hierarchy kept apart from the data, in an ontology, can be supplied to the processor (section 05.01.4.3 of the [validation guide](05.01-validating-with-shacl-1.2.ipynb)). A hierarchy defined in the shapes graph is not read by default. However, the ontology and the shapes graph may be held in the same graph, such as when the resources are defined as `sh:ShapeClass`. 

SHACL 1.2 says processors SHOULD offer a parameter that reads `rdfs:subClassOf` from the shapes graph as well. In StarLayer it is the option `subclass_of_in_shapes_graph=True`, a setting of the processor or of a single call. Note that `rdf:type` triples, which declare a resource to be a member of a class, are always read from the data graph.

Below, `ex:Dog` is a subclass of `ex:Pet` in the shapes graph. The data says that `ex:rex` is a dog, with no name. By default `ex:PetShape`, which targets pets (and requires a name), does not reach `ex:rex` (conforms); but with the option set to true it does, and `ex:rex` does not conform.

In [4]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Dog rdfs:subClassOf ex:Pet .
    ex:PetShape a sh:NodeShape ;
      sh:targetClass ex:Pet ;
      sh:property [ sh:path ex:name ; sh:minCount 1 ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:rex a ex:Dog .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)
print("default:", myProcessor.validate(data_graph=data).conforms)
print("")
print("for this call:", myProcessor.validate(data_graph=data, subclass_of_in_shapes_graph=True).conforms)

# the same setting for every call of a processor
shapesAwareProcessor = StarShaclProcessor(shacl_graph=shapes, subclass_of_in_shapes_graph=True)
print(shapesAwareProcessor.validate(data_graph=data).report_text)

default: True

for this call: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:MinCountConstraintComponent:
	Severity: sh:Violation
	Source Shape: [ sh:minCount 1 ; sh:path ex:name ]
	Focus Node: ex:rex
	Result Path: ex:name
	Message: Needs at least 1 value for ex:name



## 05.03.3 Properties inherited through the class hierarchy: `sh:closed sh:ByTypes`

`sh:closed true` says that a node may have only the properties the shape lists with `sh:property`, plus any in `sh:ignoredProperties`. Every other property of the node is a violation.

`sh:closed sh:ByTypes` is more flexible. It builds the allowed set of properties from the node's own class membership (its `rdf:type` statements). The node's types identify the relevant shapes, and the properties those shapes list become allowed properties of the node.

The relevant shapes are:

- shapes that are also classes: a type of the node, or a superclass of one,
- shapes that target a type of the node, or a superclass of one, with `sh:targetClass`, and
- shapes reached from either of these through `sh:node`.

Points to consider:

- The node's own types come from the data graph (`ex:rex rdf:type ex:Dog`).
- The classes (`ex:Dog a rdfs:Class` or `ex:Dog a sh:ShapeClass`) and the hierarchy (`ex:Dog rdfs:subClassOf ex:Animal`) are read from the shapes graph. A type that is not declared a class in the shapes graph adds no properties.  This allows the shapes graph to control class membership solely for purposes of inherited properties.
- The closed shape's own properties count only when it is a relevant shape. A closed shape that is not itself a class must target one of the node's types or superclasses with `sh:targetClass`, or be reached through `sh:node`.
- Only a `sh:path` that is a plain property IRI adds a property. A complex path (a sequence, inverse, alternative or `*` path) adds nothing.

`rdf:type` is always allowed as a property, as is anything in the `sh:ignoredProperties` of the closed shape. The `sh:ignoredProperties` of the superclass shapes are not inherited.

Below, `ex:Dog` is the closed shape. `ex:Dog` and `ex:Animal` are each both a class and a node shape (section 05.03.1), and `ex:Dog` is a subclass of `ex:Animal` and of `ex:Pet`. `ex:Pet` is declared a class in the shapes graph, and `ex:PetShape` is a separate shape that targets `ex:Pet` with `sh:targetClass`. A dog may therefore have `ex:name`, listed by `ex:Animal`, `ex:barks`, listed by `ex:Dog`, and `ex:owner`, listed by `ex:PetShape`, and nothing else.

`ex:rex` and `ex:fido` are both typed `ex:Dog` in the data graph. The class hierarchy in the shapes graph makes `ex:Animal` and `ex:Pet` count as their types too.

`ex:rex` has only the allowed properties (conforms), and `ex:fido` also has `ex:meows`, which none of its types declare (does not conform).

In [3]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Animal a rdfs:Class, sh:NodeShape ;
      sh:property [ sh:path ex:name ] .
    ex:Dog a rdfs:Class, sh:NodeShape ;
      rdfs:subClassOf ex:Animal, ex:Pet ;
      sh:closed sh:ByTypes ;
      sh:property [ sh:path ex:barks ] .
    ex:Pet a rdfs:Class .
    ex:PetShape a sh:NodeShape ;
      sh:targetClass ex:Pet ;
      sh:property [ sh:path ex:owner ] .
""", format="turtle")

# the processor is created from the shapes graph, then used to validate the data graph
myProcessor = StarShaclProcessor(shacl_graph=shapes)

good = StarLayerGraph()
good.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:rex a ex:Dog ; ex:name "Rex" ; ex:barks "woof" ; ex:owner "Sam" .
""", format="turtle")
print("conforms:", myProcessor.validate(data_graph=good).conforms)

bad = StarLayerGraph()
bad.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:fido a ex:Dog ; ex:name "Fido" ; ex:meows "meow" .
""", format="turtle")
result = myProcessor.validate(data_graph=bad)
print("conforms:", result.conforms)
print(result.report_text)

conforms: True
conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in sh:ClosedConstraintComponent:
	Severity: sh:Violation
	Source Shape: ex:Dog
	Focus Node: ex:fido
	Value Node: "meow"
	Result Path: ex:meows
	Message: ex:fido has the property ex:meows, which none of its types declare

